# LAV-DF Audio Deepfake Detection — ResNet18 & AASIST

**Reproducible experiment notebook (curated from the original Kaggle exploration).** This notebook documents preprocessing, model training, evaluation, and Integrated Gradients/acoustic visualisation. Code is retained from the source notebook where possible; exploratory file searches, repeated publishing commands, and debugging have been omitted.

**Important:** Run sections in order, configure the paths below, and verify the required supporting scripts. Training is expensive; XAI sections require trained checkpoints and prepared audio. This notebook has not been executed in this environment. No results are fabricated.

**Label convention:** `1 = audio manipulated`, `0 = audio authentic` (including visual-only manipulation). The task is **audio manipulation detection**, not general audiovisual deepfake classification.

## 1. Environment and dataset paths
Set these paths for your own Kaggle datasets. The upstream project requires `audio_data.py`, `common.py`, and the official AASIST implementation in addition to the uploaded files.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import torch

DATA_ROOT = Path('/kaggle/input/datasets/gohminghan/lav-df-5000/LAV-DF-5000')
PROJECT_ROOT = Path('/kaggle/input/datasets/gohminghan/experiment-2a/experiment_2a')
AUDIO_ROOT = Path('/kaggle/working/audio_out_full')
RUN_ROOT = Path('/kaggle/working/lavdf_runs')
AASIST_DIR = PROJECT_ROOT / 'third_party' / 'aasist'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
for name, path in {'data':DATA_ROOT,'project':PROJECT_ROOT}.items():
    print(name, path, 'exists:', path.exists())
for name in ('prepare_audio.py','experiment.py','models.py','audio_data.py','common.py'):
    print(name, (PROJECT_ROOT / name).is_file())


## Project source modules and dependency checks

The notebook uses the original Python modules rather than duplicating their implementations:

- `prepare_audio.py`: metadata validation, FFmpeg extraction, 16 kHz waveform windows and log-Mel feature caching.
- `audio_data.py`: aligned waveform and spectrogram datasets for AASIST and ResNet18.
- `common.py`: train-only normalization, data loading, deterministic seeding, threshold selection and evaluation metrics.
- `models.py`: single-channel ResNet18 and the AASIST wrapper.
- `experiment(1).py`: training, evaluation and feature export entry points.

Keep these modules together in your repository. AASIST also requires its third-party implementation (`third_party/aasist/AASIST.py` and `AASIST.conf`) which is **not** included in the supplied files. Check its license and attribution before publishing. Raw LAV-DF media and pretrained weights should be obtained separately under their respective terms.

**Execution status:** The notebook is organised and documented but the full Kaggle workflow has not been rerun. In particular, the XAI cells retain original experiment-specific paths and require configuration before use.

In [ ]:
# Check that the source modules are present without running training.
from pathlib import Path
required_modules = ["prepare_audio.py", "audio_data.py", "common.py", "models.py", "experiment(1).py"]
module_status = {name: (PROJECT_ROOT / name).is_file() for name in required_modules}
for name, present in module_status.items():
    print(f"{'OK' if present else 'MISSING'}: {name}")
if not all(module_status.values()):
    print("Set PROJECT_ROOT to the folder containing all source modules before proceeding.")

## 2. Preprocessing — matched audio representations
The source preprocessing script extracts 16 kHz mono audio, uses 4.0375-second (64,600-sample) windows with 0.5-second stride, and generates 128-bin log-Mel spectrograms. ResNet18 receives spectrograms; AASIST receives waveforms. Window labels require overlap with an annotated **audio** manipulation interval. The preprocessing script checks audio/video timing and stores a manifest.

**Run only if the processed audio dataset has not already been prepared.** The command may take considerable time and disk space.

In [ ]:
# Uncomment to preprocess from the raw LAV-DF videos.
# subprocess.run([
#     sys.executable, str(PROJECT_ROOT / 'prepare_audio.py'),
#     '--data-root', str(DATA_ROOT), '--out', str(AUDIO_ROOT),
#     '--splits', 'train', 'dev', 'test'
# ], check=True)


### Validate prepared dataset

In [ ]:
# If preprocessing was published as a Kaggle dataset, change AUDIO_ROOT above
# to that dataset's mounted directory before running the following cells.
for filename in ('config.json', 'manifest.jsonl', 'summary.json'):
    path = AUDIO_ROOT / filename
    print(f'{filename}: {path.exists()}')
if (AUDIO_ROOT / 'summary.json').is_file():
    print(json.dumps(json.loads((AUDIO_ROOT / 'summary.json').read_text()), indent=2))


## 3. Training protocol
The provided `experiment(1).py` implements AdamW, a cosine learning-rate scheduler, inverse-class-frequency sampling, BCE-with-logits loss, gradient clipping, validation video-level ROC-AUC checkpoint selection, and early stopping. The project used a **maximum of 100 epochs**; the script default is 30 unless overridden. Specify actual run settings rather than assuming defaults. Output folders must be new/empty.

### 3.1 ResNet18 training
Single-channel log-Mel inputs feed a modified ResNet18 backbone and a 512-dimensional embedding with dropout (0.2) and binary classification head. The following is a **template**: verify against the original run configuration before claiming exact reproducibility.

In [ ]:
# Uncomment when ready to train. Check the actual original run settings.
# subprocess.run([
#     sys.executable, str(PROJECT_ROOT / 'experiment.py'), 'train',
#     '--root', str(AUDIO_ROOT), '--model', 'resnet18',
#     '--init', 'pretrained', '--out', str(RUN_ROOT / 'resnet18_pretrained_clean'),
#     '--epochs', '100', '--seed', '42'
# ], check=True)


### 3.2 AASIST training
AASIST uses 64,600-sample raw waveforms. Its official architecture/configuration and pretrained weights must be available separately. The following is a **template**, not a verified record of the completed training run.

In [ ]:
AASIST_WEIGHTS = AASIST_DIR / 'AASIST.pth'  # adjust to your actual checkpoint path
# subprocess.run([
#     sys.executable, str(PROJECT_ROOT / 'experiment.py'), 'train',
#     '--root', str(AUDIO_ROOT), '--model', 'aasist',
#     '--init', 'pretrained', '--aasist-dir', str(AASIST_DIR),
#     '--aasist-weights', str(AASIST_WEIGHTS),
#     '--out', str(RUN_ROOT / 'aasist_pretrained_clean'),
#     '--epochs', '100', '--seed', '42'
# ], check=True)


## 4. Evaluation and checkpoint metadata
Evaluate on the held-out test split using the development-selected checkpoint and thresholds. Do not use test data to tune thresholds. The model script exports video- and window-level predictions.

In [ ]:
# Point to actual existing checkpoints. Avoid executing if absent.
CHECKPOINTS = {
    'ResNet18': RUN_ROOT / 'resnet18_pretrained' / 'best.pt',
    'AASIST': RUN_ROOT / 'E2A_aasist_pretrained_s42' / 'best.pt',
}
for model_name, checkpoint_path in CHECKPOINTS.items():
    print(model_name, checkpoint_path, 'exists:', checkpoint_path.is_file())
    if checkpoint_path.is_file():
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        print('  selected epoch:', checkpoint.get('epoch'))
        print('  development ROC-AUC:', checkpoint.get('dev_video_audio_roc_auc'))
        print('  protocol:', checkpoint.get('protocol'))


In [ ]:
# Example held-out evaluation; uncomment when AUDIO_ROOT and checkpoints are configured.
# for name, ckpt in CHECKPOINTS.items():
#     subprocess.run([
#         sys.executable, str(PROJECT_ROOT / 'experiment.py'), 'evaluate',
#         '--root', str(AUDIO_ROOT), '--checkpoint', str(ckpt),
#         '--split', 'test', '--out', str(RUN_ROOT / f'{name.lower()}_test_metrics')
#     ], check=True)


## 5. Explainability — Integrated Gradients and acoustic analysis
The original notebook generates Integrated Gradients (IG) and acoustic figures for both models, using selected test windows. The following cells preserve the original analysis logic. **Run ResNet18 before AASIST**, because the AASIST section reuses selected windows and the sample CSV from ResNet18. Adjust paths to the mounted checkpoints and prepared test manifest. IG attribution is not itself proof of accurate temporal localisation.

The original XAI cells may install Captum and require `pandas`, `matplotlib`, `librosa`, `soundfile`, and the model implementation. They have not been independently validated here.

### 5.1 ResNet18 — setup, sample selection and IG

In [ ]:
!pip install -q captum librosa soundfile


import sys
import json
import random
import shutil
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

from captum.attr import IntegratedGradients
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Automatically find the original Experiment 2A project
candidates = [
    p.parent
    for p in Path("/kaggle/input").rglob("models.py")
    if (p.parent / "experiment.py").is_file()
    and (p.parent / "prepare_audio.py").is_file()
    and (p.parent / "trained_models/resnet18_pretrained/best.pt").is_file()
]

if not candidates:
    raise FileNotFoundError(
        "Original Experiment 2A project not found. "
        "Check that Version 10 is attached."
    )

PROJECT_ROOT = candidates[0]

VIDEO_ROOT = PROJECT_ROOT / "LAV-DF-5000/LAV-DF-5000"
CHECKPOINT_PATH = PROJECT_ROOT / "trained_models/resnet18_pretrained/best.pt"

OUTPUT_ROOT = Path("/kaggle/working/ig_experiment")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

from models import AudioResNet18
from common import SpectrogramDataset
from prepare_audio import repeat_pad

print("Project root:", PROJECT_ROOT)
print("Video root:", VIDEO_ROOT)
print("Device:", device)
print("Output:", OUTPUT_ROOT)

assert CHECKPOINT_PATH.is_file()
assert VIDEO_ROOT.is_dir()


In [ ]:

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

model = AudioResNet18(pretrained=False)
model.load_state_dict(checkpoint["model"], strict=True)
model = model.to(device)
model.eval()

print("Model loaded:", type(model).__name__)
print("Target:", checkpoint["target"])
print("Sample rate:", checkpoint["audio"]["sample_rate"])
print("Window duration:", checkpoint["audio"]["window_seconds"])
print("Mel bins:", checkpoint["audio"]["n_mels"])


In [ ]:

TEST_MANIFEST = VIDEO_ROOT / "test_manifest.jsonl"

with TEST_MANIFEST.open("r", encoding="utf-8") as f:
    test_records = [
        json.loads(line)
        for line in f if line.strip()
    ]

fake_pool = [
    r for r in test_records
    if r["audio_video_label"] == 1
    and (VIDEO_ROOT / r["file"]).is_file()
]

real_pool = [
    r for r in test_records
    if r["audio_video_label"] == 0
    and (VIDEO_ROOT / r["file"]).is_file()
]

assert len(fake_pool) >= 5
assert len(real_pool) >= 5

rng = random.Random(SEED)

selected = (
    rng.sample(fake_pool, 5)
    + rng.sample(real_pool, 5)
)

print("Selected videos:")
for r in selected:
    print(r["file"], "FAKE" if r["audio_video_label"] else "REAL")


In [ ]:

metadata_candidates = [
    VIDEO_ROOT / "metadata.min.json",
    VIDEO_ROOT / "metadata.json"
]

metadata_path = next(
    (p for p in metadata_candidates if p.is_file()),
    None
)

if metadata_path is None:
    raise FileNotFoundError("Official LAV-DF metadata not found")

with metadata_path.open("r", encoding="utf-8") as f:
    original_metadata = json.load(f)

selected_names = {r["file"] for r in selected}

selected_metadata = [
    r for r in original_metadata
    if r["file"] in selected_names
]

assert len(selected_metadata) == 10

METADATA_PATH = OUTPUT_ROOT / "selected_metadata.json"

with METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(selected_metadata, f, indent=2)

PREPARED_ROOT = OUTPUT_ROOT / "prepared_audio"

audio_cfg = checkpoint["audio"]

command = [
    sys.executable,
    str(PROJECT_ROOT / "prepare_audio.py"),
    "--data-root", str(VIDEO_ROOT),
    "--metadata", str(METADATA_PATH),
    "--out", str(PREPARED_ROOT),
    "--splits", "test",
    "--window-seconds", str(audio_cfg["window_seconds"]),
    "--stride-seconds", str(audio_cfg["stride_seconds"]),
    "--sample-rate", str(audio_cfg["sample_rate"]),
    "--n-mels", str(audio_cfg["n_mels"]),
    "--n-fft", str(audio_cfg["n_fft"]),
    "--win-length", str(audio_cfg["win_length"]),
    "--hop-length", str(audio_cfg["hop_length"]),
    "--fmax", str(audio_cfg["fmax"]),
    "--storage-dtype", audio_cfg["storage_dtype"]
]

subprocess.run(command, check=True)

print("Preprocessing completed.")


In [ ]:

manifest_path = PREPARED_ROOT / "manifest.jsonl"

with manifest_path.open("r", encoding="utf-8") as f:
    prepared_records = [
        json.loads(line)
        for line in f if line.strip()
    ]

assert len(prepared_records) == 10

labels = Counter(r["audio_video_label"] for r in prepared_records)
assert labels == Counter({0: 5, 1: 5})

# Verify preprocessing compatibility
assert all(
    r["config_id"] == checkpoint["config_id"]
    for r in prepared_records
), "Preprocessing config mismatch. Do not bypass this check."

test_dataset = SpectrogramDataset(
    root=PREPARED_ROOT,
    records=prepared_records,
    split="test",
    stats=checkpoint["stats"],
    audio=checkpoint["audio"]
)

print("Prepared videos:", len(prepared_records))
print("Prepared windows:", len(test_dataset))


In [ ]:

selected_windows = []

for record_idx, record in enumerate(test_dataset.records):

    windows = record["windows"]

    if record["audio_video_label"] == 1:
        window_idx = max(
            range(len(windows)),
            key=lambda j: windows[j]["audio_fake_seconds"]
        )
    else:
        window_idx = 0

    dataset_idx = next(
        k for k, (i, j) in enumerate(test_dataset.index)
        if i == record_idx and j == window_idx
    )

    selected_windows.append({
        "record": record,
        "window_idx": window_idx,
        "dataset_idx": dataset_idx
    })

for s in selected_windows:
    print(
        s["record"]["file"],
        "| window:", s["window_idx"],
        "| label:", s["record"]["audio_video_label"]
    )


In [ ]:

ig = IntegratedGradients(model)
results = []

for i, sample in enumerate(selected_windows, start=1):

    record = sample["record"]
    window_idx = sample["window_idx"]
    window = record["windows"][window_idx]

    x, label, _ = test_dataset[sample["dataset_idx"]]
    x = x.unsqueeze(0).float().to(device)

    with torch.no_grad():
        logit = model(x).reshape(-1)
        fake_prob = torch.sigmoid(logit)[0].item()

    attr, delta = ig.attribute(
        inputs=x,
        baselines=torch.zeros_like(x),
        n_steps=64,
        internal_batch_size=8,
        method="gausslegendre",
        return_convergence_delta=True
    )

    results.append({
        "file": record["file"],
        "true_label": int(label.item()),
        "predicted_label": int(fake_prob >= 0.5),
        "fake_probability": fake_prob,
        "convergence_delta": float(delta.abs().max().item()),
        "spectrogram": x.detach().cpu().numpy()[0, 0],
        "attribution": attr.detach().cpu().numpy()[0, 0],
        "window_index": window_idx,
        "window_start": window["start_s"],
        "window_end": window["end_s"],
        "fake_periods": record.get("fake_periods", [])
    })

    print(
        f"[{i}/10] {record['file']} | "
        f"P(fake)={fake_prob:.4f}"
    )

print("Integrated Gradients completed for 10 samples.")


In [ ]:

IG_DIR = OUTPUT_ROOT / "IG_figures"
IG_DIR.mkdir(parents=True, exist_ok=True)

duration = checkpoint["audio"]["window_seconds"]

for i, r in enumerate(results, start=1):

    spec = r["spectrogram"]
    attr = r["attribution"]

    limit = max(np.percentile(np.abs(attr), 99), 1e-10)
    extent = [0, duration, 0, spec.shape[0]]

    fig, axes = plt.subplots(
        3, 1, figsize=(14, 10), sharex=True
    )

    axes[0].imshow(
        spec, origin="lower", aspect="auto",
        extent=extent, cmap="magma"
    )
    axes[0].set_title("Normalized Log-Mel Spectrogram")

    im = axes[1].imshow(
        attr, origin="lower", aspect="auto",
        extent=extent, cmap="seismic",
        vmin=-limit, vmax=limit
    )
    axes[1].set_title("Integrated Gradients Attribution")
    fig.colorbar(im, ax=axes[1], label="Attribution to fake logit")

    axes[2].imshow(
        spec, origin="lower", aspect="auto",
        extent=extent, cmap="gray"
    )
    axes[2].imshow(
        attr, origin="lower", aspect="auto",
        extent=extent, cmap="seismic",
        vmin=-limit, vmax=limit, alpha=0.5
    )
    axes[2].set_title("Integrated Gradients Overlay")

    for start, end in r["fake_periods"]:
        a = max(0, start - r["window_start"])
        b = min(duration, end - r["window_start"])
        if b > a:
            for ax in axes:
                ax.axvspan(a, b, color="lime", alpha=0.2)

    for ax in axes:
        ax.set_ylabel("Mel bin")
    axes[-1].set_xlabel("Time within window (seconds)")

    label = "FAKE" if r["true_label"] else "REAL"

    fig.suptitle(
        f"{r['file']} | True: {label} | "
        f"P(fake): {r['fake_probability']:.4f}"
    )

    fig.tight_layout()

    fig.savefig(
        IG_DIR / f"sample_{i:02d}_{label.lower()}_ig.png",
        dpi=180, bbox_inches="tight"
    )
    plt.close(fig)

print("Saved IG figures:", len(list(IG_DIR.glob("*.png"))))


### 5.2 ResNet18 — acoustic visualisations and export

In [ ]:

ACOUSTIC_DIR = OUTPUT_ROOT / "Acoustic_figures"
ACOUSTIC_DIR.mkdir(parents=True, exist_ok=True)

SR = checkpoint["audio"]["sample_rate"]
HOP = checkpoint["audio"]["hop_length"]
N_FFT = checkpoint["audio"]["n_fft"]
WIN = checkpoint["audio"]["win_length"]
WINDOW_SAMPLES = round(duration * SR)

acoustic_rows = []

for i, r in enumerate(results, start=1):

    record = next(
        row for row in test_dataset.records
        if row["file"] == r["file"]
    )
    window = record["windows"][r["window_index"]]

    wav_path = PREPARED_ROOT / record["wav_path"]
    y_full, sr = sf.read(wav_path, dtype="float32")

    assert sr == SR and y_full.ndim == 1

    y = y_full[
        window["start_sample"]:window["end_sample"]
    ]

    if len(y) != WINDOW_SAMPLES:
        y = repeat_pad(y, WINDOW_SAMPLES)

    S = np.abs(librosa.stft(
        y, n_fft=N_FFT, win_length=WIN,
        hop_length=HOP
    ))

    times = librosa.frames_to_time(
        np.arange(S.shape[1]), sr=SR, hop_length=HOP
    )

    rms = librosa.feature.rms(
        y=y, frame_length=N_FFT, hop_length=HOP
    )[0]

    flatness = librosa.feature.spectral_flatness(S=S)[0]

    Sn = S / (S.sum(axis=0, keepdims=True) + 1e-10)
    flux = np.pad(
        np.sqrt(np.sum(np.diff(Sn, axis=1)**2, axis=0)),
        (1, 0)
    )

    f0, voiced, voiced_prob = librosa.pyin(
        y, sr=SR, fmin=65, fmax=500,
        frame_length=1024, hop_length=HOP
    )

    pitch_times = librosa.frames_to_time(
        np.arange(len(f0)), sr=SR, hop_length=HOP
    )

    fig, axes = plt.subplots(
        4, 1, figsize=(14, 11), sharex=True
    )

    for ax, values, title, ylabel in zip(
        axes[:3],
        [rms, flux, flatness],
        ["RMS Energy", "Spectral Flux", "Spectral Flatness"],
        ["Amplitude", "Flux", "Flatness"]
    ):
        ax.plot(times[:len(values)], values)
        ax.set_title(title)
        ax.set_ylabel(ylabel)

    axes[3].plot(pitch_times, f0)
    axes[3].set_title("Fundamental Frequency (Voiced Only)")
    axes[3].set_ylabel("Pitch (Hz)")
    axes[3].set_xlabel("Time within window (seconds)")

    for start, end in r["fake_periods"]:
        a = max(0, start - r["window_start"])
        b = min(duration, end - r["window_start"])
        if b > a:
            for ax in axes:
                ax.axvspan(a, b, color="orange", alpha=0.2)

    for ax in axes:
        ax.set_xlim(0, duration)
        ax.grid(alpha=0.2)

    label = "FAKE" if r["true_label"] else "REAL"

    fig.suptitle(
        f"{r['file']} | True: {label} | "
        f"P(fake): {r['fake_probability']:.4f}"
    )

    fig.tight_layout()
    fig.savefig(
        ACOUSTIC_DIR /
        f"sample_{i:02d}_{label.lower()}_acoustic.png",
        dpi=180, bbox_inches="tight"
    )
    plt.close(fig)

    valid_f0 = f0[np.isfinite(f0)]

    acoustic_rows.append({
        "video": r["file"],
        "true_label": r["true_label"],
        "fake_probability": r["fake_probability"],
        "mean_rms": float(np.mean(rms)),
        "mean_flux": float(np.mean(flux)),
        "mean_flatness": float(np.mean(flatness)),
        "median_pitch_hz": (
            float(np.median(valid_f0))
            if len(valid_f0) else np.nan
        ),
        "voiced_fraction": float(np.mean(voiced))
    })

print("Saved acoustic figures:", len(list(ACOUSTIC_DIR.glob("*.png"))))


In [ ]:

FULL_DIR = OUTPUT_ROOT / "Test_audio/full_clips"
WINDOW_DIR = OUTPUT_ROOT / "Test_audio/selected_windows"
NPZ_DIR = OUTPUT_ROOT / "Test_audio/spectrograms"
SUMMARY_DIR = OUTPUT_ROOT / "Summary"

for folder in [FULL_DIR, WINDOW_DIR, NPZ_DIR, SUMMARY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

sample_rows = []

for i, r in enumerate(results, start=1):

    record = next(
        row for row in test_dataset.records
        if row["file"] == r["file"]
    )
    window = record["windows"][r["window_index"]]

    label = "fake" if r["true_label"] else "real"
    name = f"sample_{i:02d}_{label}"

    wav_source = PREPARED_ROOT / record["wav_path"]
    npz_source = PREPARED_ROOT / record["feature_path"]

    shutil.copy2(wav_source, FULL_DIR / f"{name}.wav")
    shutil.copy2(npz_source, NPZ_DIR / f"{name}.npz")

    y, sr = sf.read(wav_source, dtype="float32")

    segment = y[
        window["start_sample"]:window["end_sample"]
    ]

    if len(segment) != WINDOW_SAMPLES:
        segment = repeat_pad(segment, WINDOW_SAMPLES)

    sf.write(
        WINDOW_DIR / f"{name}.wav",
        segment, SR, subtype="FLOAT"
    )

    sample_rows.append({
        "sample_id": name,
        "video": r["file"],
        "true_label": r["true_label"],
        "predicted_label_resnet18": r["predicted_label"],
        "fake_probability_resnet18": r["fake_probability"],
        "window_index": r["window_index"],
        "window_start_s": r["window_start"],
        "window_end_s": r["window_end"],
        "fake_periods": json.dumps(r["fake_periods"]),
        "ig_convergence_delta": r["convergence_delta"]
    })

pd.DataFrame(sample_rows).to_csv(
    SUMMARY_DIR / "selected_samples.csv",
    index=False
)

pd.DataFrame(acoustic_rows).to_csv(
    SUMMARY_DIR / "acoustic_summary.csv",
    index=False
)

print("Saved AASIST windows:", len(list(WINDOW_DIR.glob("*.wav"))))


In [ ]:
checks = {
    "IG figures": len(list(IG_DIR.glob("*.png"))),
    "Acoustic figures": len(list(ACOUSTIC_DIR.glob("*.png"))),
    "Full audio": len(list(FULL_DIR.glob("*.wav"))),
    "AASIST windows": len(list(WINDOW_DIR.glob("*.wav"))),
    "Spectrograms": len(list(NPZ_DIR.glob("*.npz")))
}

for name, count in checks.items():
    print(f"{name}: {count}/10")
    assert count == 10

for wav in WINDOW_DIR.glob("*.wav"):
    info = sf.info(wav)
    assert info.samplerate == 16000
    assert info.frames == 64600
    assert info.channels == 1

print("\nAll 10 samples and analyses verified.")
print("Saved to:", OUTPUT_ROOT)

### 5.3 AASIST — setup, IG and acoustic visualisations

In [ ]:
!pip install -q captum librosa soundfile


import sys
import json
import gc
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

from captum.attr import IntegratedGradients
from IPython.display import display, Image, Markdown

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Reuse the project root from the ResNet experiment
if "PROJECT_ROOT" not in globals():
    candidates = [
        p.parent
        for p in Path("/kaggle/input").rglob("models.py")
        if (p.parent / "experiment.py").is_file()
        and (p.parent / "prepare_audio.py").is_file()
    ]
    if not candidates:
        raise FileNotFoundError("Experiment 2A project not found")
    PROJECT_ROOT = candidates[0]

sys.path.insert(0, str(PROJECT_ROOT))

from models import AudioAASIST

RESNET_ROOT = Path("/kaggle/working/ig_experiment")
AASIST_ROOT = Path("/kaggle/working/aasist_ig_experiment")

IG_DIR = AASIST_ROOT / "IG_figures"
ACOUSTIC_DIR = AASIST_ROOT / "Acoustic_figures"
SUMMARY_DIR = AASIST_ROOT / "Summary"

for folder in [IG_DIR, ACOUSTIC_DIR, SUMMARY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Device:", device)
print("AASIST output:", AASIST_ROOT)


In [ ]:

AASIST_CHECKPOINT = (
    PROJECT_ROOT / "trained_models/aasist_pretrained/best.pt"
)

assert AASIST_CHECKPOINT.is_file()

aasist_checkpoint = torch.load(
    AASIST_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint keys:")
print(list(aasist_checkpoint.keys()))

# Construct original architecture.
# Initialization weights are unnecessary because the
# trained checkpoint will overwrite all parameters.
aasist_model = AudioAASIST(weights=None)

aasist_model.load_state_dict(
    aasist_checkpoint["model"],
    strict=True
)

aasist_model = aasist_model.to(device)
aasist_model.eval()

print("AASIST loaded successfully!")
print("Target:", aasist_checkpoint.get("target"))


In [ ]:

WINDOW_DIR = RESNET_ROOT / "Test_audio/selected_windows"
SAMPLE_CSV = RESNET_ROOT / "Summary/selected_samples.csv"

assert WINDOW_DIR.is_dir(), "ResNet audio windows missing"
assert SAMPLE_CSV.is_file(), "ResNet sample mapping missing"

sample_df = pd.read_csv(SAMPLE_CSV)

assert len(sample_df) == 10
assert sample_df["true_label"].value_counts().to_dict() == {
    0: 5, 1: 5
}

audio_samples = []

for _, row in sample_df.iterrows():

    wav_path = WINDOW_DIR / f"{row['sample_id']}.wav"

    if not wav_path.is_file():
        raise FileNotFoundError(wav_path)

    y, sr = sf.read(wav_path, dtype="float32")

    assert sr == 16000
    assert y.ndim == 1
    assert len(y) == 64600

    audio_samples.append({
        "sample_id": row["sample_id"],
        "video": row["video"],
        "true_label": int(row["true_label"]),
        "window_start": float(row["window_start_s"]),
        "window_end": float(row["window_end_s"]),
        "waveform": y
    })

print("Loaded audio samples:", len(audio_samples))


In [ ]:

aasist_model.eval()

ig = IntegratedGradients(aasist_model)

aasist_results = []

for i, sample in enumerate(audio_samples, start=1):

    x = torch.from_numpy(
        sample["waveform"]
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        logit = aasist_model(x).reshape(-1)
        fake_prob = torch.sigmoid(logit)[0].item()

    # Zero waveform represents silence
    baseline = torch.zeros_like(x)

    attr, delta = ig.attribute(
        inputs=x,
        baselines=baseline,
        n_steps=32,
        internal_batch_size=1,
        method="gausslegendre",
        return_convergence_delta=True
    )

    aasist_results.append({
        **sample,
        "fake_probability": fake_prob,
        "predicted_label": int(fake_prob >= 0.5),
        "attribution": attr.detach().cpu().numpy()[0],
        "convergence_delta": float(
            delta.detach().abs().max().item()
        )
    })

    print(
        f"[{i}/10] {sample['video']} | "
        f"True={sample['true_label']} | "
        f"Pred={int(fake_prob >= 0.5)} | "
        f"P(fake)={fake_prob:.4f}"
    )

    del x, attr, delta
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("AASIST Integrated Gradients completed!")


In [ ]:

SR = 16000
DURATION = 64600 / SR

for i, r in enumerate(aasist_results, start=1):

    y = r["waveform"]
    attr = r["attribution"]

    times = np.arange(len(y)) / SR

    # Aggregate attribution into 10 ms intervals
    hop = 160
    n = len(attr) // hop

    signed = attr[:n*hop].reshape(n, hop).sum(axis=1)
    magnitude = np.abs(
        attr[:n*hop]
    ).reshape(n, hop).sum(axis=1)

    attr_times = (
        np.arange(n) + 0.5
    ) * hop / SR

    fig, axes = plt.subplots(
        3, 1, figsize=(14, 9), sharex=True
    )

    axes[0].plot(times, y, linewidth=0.5)
    axes[0].set_title("Original Audio Waveform")
    axes[0].set_ylabel("Amplitude")

    axes[1].plot(attr_times, signed, linewidth=1)
    axes[1].axhline(0, linewidth=0.8, color="gray")
    axes[1].set_title("Signed Integrated Gradients")
    axes[1].set_ylabel("Signed attribution")

    axes[2].plot(attr_times, magnitude, linewidth=1)
    axes[2].set_title("Absolute Integrated Gradients")
    axes[2].set_ylabel("Attribution magnitude")
    axes[2].set_xlabel("Time within window (seconds)")

    for ax in axes:
        ax.set_xlim(0, DURATION)
        ax.grid(alpha=0.2)

    label = "FAKE" if r["true_label"] else "REAL"

    fig.suptitle(
        f"AASIST | {r['video']} | True: {label} | "
        f"P(fake): {r['fake_probability']:.4f}"
    )

    fig.tight_layout()

    fig.savefig(
        IG_DIR / f"{r['sample_id']}_aasist_ig.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.close(fig)

print("Saved IG figures:", len(list(IG_DIR.glob("*.png"))))


In [ ]:

HOP = 160
N_FFT = 512
WIN = 400

acoustic_rows = []

for i, r in enumerate(aasist_results, start=1):

    y = r["waveform"]

    S = np.abs(librosa.stft(
        y, n_fft=N_FFT, win_length=WIN,
        hop_length=HOP
    ))

    times = librosa.frames_to_time(
        np.arange(S.shape[1]),
        sr=SR, hop_length=HOP
    )

    rms = librosa.feature.rms(
        y=y, frame_length=N_FFT, hop_length=HOP
    )[0]

    flatness = librosa.feature.spectral_flatness(S=S)[0]

    Sn = S / (S.sum(axis=0, keepdims=True) + 1e-10)

    flux = np.pad(
        np.sqrt(np.sum(np.diff(Sn, axis=1)**2, axis=0)),
        (1, 0)
    )

    f0, voiced, voiced_prob = librosa.pyin(
        y, sr=SR, fmin=65, fmax=500,
        frame_length=1024, hop_length=HOP
    )

    pitch_times = librosa.frames_to_time(
        np.arange(len(f0)),
        sr=SR, hop_length=HOP
    )

    fig, axes = plt.subplots(
        4, 1, figsize=(14, 11), sharex=True
    )

    for ax, values, title, ylabel in zip(
        axes[:3],
        [rms, flux, flatness],
        ["RMS Energy", "Spectral Flux", "Spectral Flatness"],
        ["Amplitude", "Flux", "Flatness"]
    ):
        ax.plot(times[:len(values)], values)
        ax.set_title(title)
        ax.set_ylabel(ylabel)

    axes[3].plot(pitch_times, f0)
    axes[3].set_title("Fundamental Frequency (Voiced Only)")
    axes[3].set_ylabel("Pitch (Hz)")
    axes[3].set_xlabel("Time within window (seconds)")

    for ax in axes:
        ax.set_xlim(0, DURATION)
        ax.grid(alpha=0.2)

    label = "FAKE" if r["true_label"] else "REAL"

    fig.suptitle(
        f"AASIST | {r['video']} | True: {label} | "
        f"P(fake): {r['fake_probability']:.4f}"
    )

    fig.tight_layout()

    fig.savefig(
        ACOUSTIC_DIR /
        f"{r['sample_id']}_aasist_acoustic.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.close(fig)

    valid_f0 = f0[np.isfinite(f0)]

    acoustic_rows.append({
        "sample_id": r["sample_id"],
        "video": r["video"],
        "true_label": r["true_label"],
        "predicted_label_aasist": r["predicted_label"],
        "fake_probability_aasist": r["fake_probability"],
        "mean_rms": float(np.mean(rms)),
        "mean_flux": float(np.mean(flux)),
        "mean_flatness": float(np.mean(flatness)),
        "median_pitch_hz": (
            float(np.median(valid_f0))
            if len(valid_f0) else np.nan
        ),
        "voiced_fraction": float(np.mean(voiced))
    })

print("Saved acoustic figures:", len(list(ACOUSTIC_DIR.glob("*.png"))))


In [ ]:
# Save AASIST result and compare with ResNet 18


aasist_summary = pd.DataFrame([
    {
        "sample_id": r["sample_id"],
        "video": r["video"],
        "true_label": r["true_label"],
        "predicted_label_aasist": r["predicted_label"],
        "fake_probability_aasist": r["fake_probability"],
        "ig_convergence_delta": r["convergence_delta"]
    }
    for r in aasist_results
])

aasist_summary.to_csv(
    SUMMARY_DIR / "aasist_ig_summary.csv",
    index=False
)

pd.DataFrame(acoustic_rows).to_csv(
    SUMMARY_DIR / "aasist_acoustic_summary.csv",
    index=False
)

resnet_summary = pd.read_csv(SAMPLE_CSV)

comparison = resnet_summary.merge(
    aasist_summary,
    on=["sample_id", "video", "true_label"]
)

comparison["resnet_correct"] = (
    comparison["predicted_label_resnet18"]
    == comparison["true_label"]
)

comparison["aasist_correct"] = (
    comparison["predicted_label_aasist"]
    == comparison["true_label"]
)

comparison.to_csv(
    SUMMARY_DIR / "resnet18_vs_aasist.csv",
    index=False
)

display(comparison[[
    "sample_id",
    "true_label",
    "fake_probability_resnet18",
    "fake_probability_aasist",
    "resnet_correct",
    "aasist_correct"
]])


## 6. Results and interpretation
Inspect model-specific IG and acoustic figures alongside ground-truth intervals and predictions. Report the test cohort, selected samples, classification results and limitations. Do not infer quantitative localisation performance from the figures alone.